In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

for fname in uploaded.keys():
    print(f" Uploaded: {fname}")

df = pd.read_csv("squaretradeprod1.csv")
print(df.head())

Saving squaretradeprod1.csv to squaretradeprod1 (1).csv
 Uploaded: squaretradeprod1 (1).csv
  exporter importer  year         flow          Y            E   distw  cntg  \
0      AGO      AGO  2020  3446.758222  5141.4478    9957.4258   288.0     0   
1      AGO      ALB  2020     0.000000  5141.4478   11211.8500  5679.0     0   
2      AGO      ARE  2020   475.994691  5141.4478  199898.1300  5914.0     0   
3      AGO      ARG  2020     0.051451  5141.4478  175280.1700  7961.0     0   
4      AGO      ARM  2020     0.000000  5141.4478    4938.1909  6364.0     0   

   lang  colony  rta  home  ison_o  ison_d      ldis  
0     0       0    0     1       2       2  5.662960  
1     0       0    0     0       2       3  8.644530  
2     0       0    0     0       2       4  8.685078  
3     0       0    0     0       2       5  8.982310  
4     0       0    0     0       2       6  8.758412  


In [ ]:
!pip install numpy pandas tqdm
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:

df = pd.read_csv("squaretradeprod1.csv")
print(df.columns)
df.head()

Index(['exporter', 'importer', 'year', 'flow', 'Y', 'E', 'distw', 'cntg',
       'lang', 'colony', 'rta', 'home', 'ison_o', 'ison_d', 'ldis'],
      dtype='object')


,exporter,importer,year,flow,Y,E,distw,cntg,lang,colony,rta,home,ison_o,ison_d,ldis
0,AGO,AGO,2020,3446.758222,5141.4478,9957.4258,288.0,0,0,0,0,1,2,2,5.662960
1,AGO,ALB,2020,0.000000,5141.4478,11211.8500,5679.0,0,0,0,0,0,2,3,8.644530
2,AGO,ARE,2020,475.994691,5141.4478,199898.1300,5914.0,0,0,0,0,0,2,4,8.685078
3,AGO,ARG,2020,0.051451,5141.4478,175280.1700,7961.0,0,0,0,0,0,2,5,8.982310
4,AGO,ARM,2020,0.000000,5141.4478,4938.1909,6364.0,0,0,0,0,0,2,6,8.758412


In [ ]:

import pandas as pd
import numpy as np


fname = list(uploaded.keys())[0]
df = pd.read_csv(fname)
print("Loaded:", fname)
print("Columns:", list(df.columns))
df.head()

Loaded: squaretradeprod1 (1).csv
Columns: ['exporter', 'importer', 'year', 'flow', 'Y', 'E', 'distw', 'cntg', 'lang', 'colony', 'rta', 'home', 'ison_o', 'ison_d', 'ldis']


,exporter,importer,year,flow,Y,E,distw,cntg,lang,colony,rta,home,ison_o,ison_d,ldis
0,AGO,AGO,2020,3446.758222,5141.4478,9957.4258,288.0,0,0,0,0,1,2,2,5.662960
1,AGO,ALB,2020,0.000000,5141.4478,11211.8500,5679.0,0,0,0,0,0,2,3,8.644530
2,AGO,ARE,2020,475.994691,5141.4478,199898.1300,5914.0,0,0,0,0,0,2,4,8.685078
3,AGO,ARG,2020,0.051451,5141.4478,175280.1700,7961.0,0,0,0,0,0,2,5,8.982310
4,AGO,ARM,2020,0.000000,5141.4478,4938.1909,6364.0,0,0,0,0,0,2,6,8.758412


In [ ]:
df.rename(columns={
    "exporter": "iso_o",
    "importer": "iso_d",
    "country_o": "iso_o",
    "country_d": "iso_d",
    "flow": "Xin",
    "trade_value": "Xin",
    "distance": "distw",
    "distw_km": "distw",
    "language_common": "lang",
    "common_language": "lang",
    "rta_dummy": "rta",
    "RTA": "rta",
    "colonial_link": "colony",
    "colony_dummy": "colony"
}, inplace=True, errors="ignore")

print("Columns after renaming:", list(df.columns))

Columns after renaming: ['iso_o', 'iso_d', 'year', 'Xin', 'Y', 'E', 'distw', 'cntg', 'lang', 'colony', 'rta', 'home', 'ison_o', 'ison_d', 'ldis']


In [ ]:
if "comcur" not in df.columns:
    df["comcur"] = 0
if "lang" not in df.columns:
    # if language column missing, create zero column (no common language)
    df["lang"] = 0
if "colony" not in df.columns:
    df["colony"] = 0
if "Xin" not in df.columns:
    raise ValueError("Required column 'Xin' (trade flow) not found. Rename your flow column to 'Xin' or 'flow' and re-run Step 3.")

# home dummy (internal trade)
df["home"] = (df["iso_o"] == df["iso_d"]).astype(int)

print("Final columns for analysis:", ["iso_o","iso_d","Xin","distw","lang","colony","rta","comcur","home"])

Final columns for analysis: ['iso_o', 'iso_d', 'Xin', 'distw', 'lang', 'colony', 'rta', 'comcur', 'home']


In [ ]:
#coefficients and phi_in
b = {
    "rta": 0.6215,
    "lang": 0.33,
    "colony": 0.84,
    "home": 1.55,
    "ldis": -1.14,
    "comcur": 0.98
}

# log distance
df["ldis"] = np.log(df["distw"].astype(float))

# phi_in
df["phi_in"] = np.exp(
    b["ldis"]*df["ldis"] + b["lang"]*df["lang"] + b["colony"]*df["colony"] +
    b["rta"]*df["rta"] + b["comcur"]*df["comcur"] + b["home"]*df["home"]
)

print("phi_in computed. sample:")
print(df[["iso_o","iso_d","Xin","phi_in"]].head())

phi_in computed. sample:
  iso_o iso_d          Xin    phi_in
0   AGO   AGO  3446.758222  0.007404
1   AGO   ALB     0.000000  0.000052
2   AGO   ARE   475.994691  0.000050
3   AGO   ARG     0.051451  0.000036
4   AGO   ARM     0.000000  0.000046


In [ ]:
Y_i = df.groupby("iso_o")["Xin"].sum().rename("Y_i")
X_n = df.groupby("iso_d")["Xin"].sum().rename("X_n")
df = df.merge(Y_i, on="iso_o").merge(X_n, on="iso_d")
df["pi_in"] = df["Xin"] / df["X_n"]

print("Aggregates computed. Examples:")
print(df[["iso_o","iso_d","Xin","Y_i","X_n","pi_in"]].head())

Aggregates computed. Examples:
  iso_o iso_d          Xin          Y_i            X_n         pi_in
0   AGO   AGO  3446.758222  5045.450668    9936.382033  3.468826e-01
1   AGO   ALB     0.000000  5045.450668   11196.474379  0.000000e+00
2   AGO   ARE   475.994691  5045.450668  198439.782834  2.398686e-03
3   AGO   ARG     0.051451  5045.450668  175256.719894  2.935746e-07
4   AGO   ARM     0.000000  5045.450668    4925.087736  0.000000e+00


In [ ]:
def contraction_mapping(df, phi_col, tol=1e-6, max_iter=300):
    iso_o_list = df["iso_o"].unique()
    iso_d_list = df["iso_d"].unique()
    Omega = pd.Series(1.0, index=iso_o_list)
    Phi = pd.Series(1.0, index=iso_d_list)

    for it in range(max_iter):

        Phi_new = df.groupby("iso_d").apply(
            lambda g: (g["pi_in"] * phi_col[g.index] * Omega[g["iso_o"]].values).sum()
        )
        Phi_new = Phi_new / Phi_new.mean()
        Omega_new = df.groupby("iso_o").apply(
            lambda g: (g["pi_in"] * phi_col[g.index] * Phi_new[g["iso_d"]].values).sum()
        )
        Omega_new = Omega_new / Omega_new.mean()

        diff = max((Phi_new - Phi).abs().max(), (Omega_new - Omega).abs().max())
        Phi, Omega = Phi_new, Omega_new
        if diff < tol:
            break
    return Omega, Phi


Omega_base, Phi_base = contraction_mapping(df, df["phi_in"])
print("Contraction mapping done. Sample Omega and Phi:")
print(Omega_base.head())
print(Phi_base.head())

/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(
/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of 

Contraction mapping done. Sample Omega and Phi:
iso_o
AGO    3.730599e-09
ALB    9.092035e-05
ARE    2.738391e-02
ARG    1.916336e-06
ARM    4.691077e-07
dtype: float64
iso_d
AGO    2.039419e-07
ALB    9.265382e-05
ARE    3.193208e-03
ARG    6.498394e-08
ARM    2.771096e-07
dtype: float64


/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(


In [ ]:

mti_results = {}
for var in ["rta", "lang", "colony", "home"]:
    df[f"{var}_prime"] = 0
    phi_prime = df["phi_in"] * np.exp(b[var]*(df[f"{var}_prime"]-df[var]))
    Omega, Phi = contraction_mapping(df, df["phi_in"])
    Omega_p, Phi_p = contraction_mapping(df, phi_prime)

    df[f"Xratio_{var}"] = 1 / (
        np.exp(b[var]*(df[f"{var}_prime"]-df[var])) *
        (Omega[df["iso_o"]].values / Omega_p[df["iso_o"]].values) *
        (Phi[df["iso_d"]].values / Phi_p[df["iso_d"]].values)
    )


    domestic_ratio = df.loc[(df["iso_o"] == df["iso_d"]) & (df[var] == 1), f"Xratio_{var}"]
    member_median = 1 / domestic_ratio.median()

    if var == "home":
        nonmember_median = np.nan
    else:
        nondom_ratio = df.loc[(df["iso_o"] == df["iso_d"]) & (df[var] == 0), f"Xratio_{var}"]
        nonmember_median = 1 / nondom_ratio.median()

    mti_results[var] = {
        "MTI_member": member_median,
        "MTI_nonmember": nonmember_median
    }
    print(f"MTI ({var}) adjusted — domestic median reciprocal: {member_median:.3f}")

/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(
/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of 

MTI (rta) adjusted — domestic median reciprocal: nan


/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(
/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of 

MTI (lang) adjusted — domestic median reciprocal: 1.197


/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(
/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of 

MTI (colony) adjusted — domestic median reciprocal: nan


/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(
/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of 

MTI (home) adjusted — domestic median reciprocal: 0.006


/tmp/ipython-input-1696795129.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Phi_new = df.groupby("iso_d").apply(
/tmp/ipython-input-1696795129.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  Omega_new = df.groupby("iso_o").apply(


In [ ]:
geti_results = {}
epsilon = -5.03
vfactor = 0.3
tol = 1e-6
max_iter = 500

for var in ["rta","lang","colony","home"]:
    df[f"{var}_prime"] = 0
    phih_in = np.exp(b[var] * (df[f"{var}_prime"] - df[var]))
    wh_i = pd.Series(1.0, index=df["iso_o"].unique())
    wh_n = pd.Series(1.0, index=df["iso_d"].unique())

    for it in range(max_iter):
        # numerator per pair
        num = (wh_i[df["iso_o"]].values ** epsilon) * phih_in.values
        # denominator per destination n (reindexed to df rows)
        denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum())
        denom_per_row = denom_per_row.reindex(df["iso_d"]).values
        pihat = num / denom_per_row

        # new expenditures and wages
        xprime_n = wh_n[df["iso_d"]].values * df["X_n"].values
        wh_i1_int = df.groupby("iso_o").apply(lambda g: (g["pi_in"] * pihat[g.index] * xprime_n[g.index]).sum())
        # align indices
        wh_i1 = wh_i1_int / Y_i.reindex(wh_i1_int.index).values
        # dampen
        aligned_wh_i = wh_i.reindex(wh_i1.index).fillna(1)
        wh_i1 = aligned_wh_i * (1 + vfactor * (wh_i1 - aligned_wh_i))
        # convergence check
        if (wh_i1 - aligned_wh_i).abs().max() < tol:
            wh_i.update(wh_i1)
            break
        wh_i.update(wh_i1)

    # final pihat and Xin_prime
    num = (wh_i[df["iso_o"]].values ** epsilon) * phih_in.values
    denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum()).reindex(df["iso_d"]).values
    pihat = num / denom_per_row
    xprime_n = wh_n[df["iso_d"]].values * df["X_n"].values
    Xin_prime = pihat * df["pi_in"].values * xprime_n

    df[f"GETI_{var}"] = 1.0 / (Xin_prime / df["Xin"].values)
    df[f"Welfare_{var}"] = pihat ** (1.0/(-epsilon))

    member_geti = df.loc[df[var]==1, f"GETI_{var}"].median()
    nonmember_geti = np.nan if var=="home" else df.loc[df[var]==0, f"GETI_{var}"].median()
    member_welf = df.loc[df[var]==1, f"Welfare_{var}"].median()
    nonmember_welf = np.nan if var=="home" else df.loc[df[var]==0, f"Welfare_{var}"].median()

    geti_results[var] = {
        "GETI_member": member_geti,
        "GETI_nonmember": nonmember_geti,
        "Welf_member": member_welf,
        "Welf_nonmember": nonmember_welf
    }
    print(f"GETI done for {var}: GETI_member={member_geti}, GETI_nonmember={nonmember_geti}, Welf_member={member_welf}")

/tmp/ipython-input-3798760240.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum())
/tmp/ipython-input-3798760240.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wh_i1_int = df.groupby("iso_o").apply(lambda g: (g["pi_in"] * pihat[g.index] * xprime_n[g.index]).sum())
/tmp/ipython-input-3798760240.py:17: DeprecationWarn

GETI done for rta: GETI_member=1.5690854436509118, GETI_nonmember=0.9124697634814594, Welf_member=0.9143648434575111


/tmp/ipython-input-3798760240.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum())
/tmp/ipython-input-3798760240.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wh_i1_int = df.groupby("iso_o").apply(lambda g: (g["pi_in"] * pihat[g.index] * xprime_n[g.index]).sum())
/tmp/ipython-input-3798760240.py:17: DeprecationWarn

GETI done for lang: GETI_member=1.335589953816457, GETI_nonmember=0.9861727233497711, Welf_member=0.9442316546812538


/tmp/ipython-input-3798760240.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wh_i1_int = df.groupby("iso_o").apply(lambda g: (g["pi_in"] * pihat[g.index] * xprime_n[g.index]).sum())
/tmp/ipython-input-3798760240.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum())
/tmp/ipython-input-3798760240.py:23: DeprecationWarn

GETI done for colony: GETI_member=2.2903588029365634, GETI_nonmember=0.9976647942169415, Welf_member=0.8480254966654335


/tmp/ipython-input-3798760240.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum())
/tmp/ipython-input-3798760240.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wh_i1_int = df.groupby("iso_o").apply(lambda g: (g["pi_in"] * pihat[g.index] * xprime_n[g.index]).sum())
/tmp/ipython-input-3798760240.py:17: DeprecationWarn

GETI done for home: GETI_member=2.5816486021792855, GETI_nonmember=nan, Welf_member=0.828155523665363


/tmp/ipython-input-3798760240.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  denom_per_row = df.groupby("iso_d").apply(lambda g: (g["pi_in"] * num[g.index]).sum())
/tmp/ipython-input-3798760240.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wh_i1_int = df.groupby("iso_o").apply(lambda g: (g["pi_in"] * pihat[g.index] * xprime_n[g.index]).sum())
/tmp/ipython-input-3798760240.py:37: DeprecationWarn

In [ ]:
rows = []
for v in ["rta","lang","colony","home"]:
    row = {"Variable": v}
    row.update(mti_results.get(v, {}))
    row.update(geti_results.get(v, {}))
    rows.append(row)

results_df = pd.DataFrame(rows)
print("\nFinal results:")
print(results_df)


outpath = "/content/PTI_MTI_GETI_Welfare_results.csv"
results_df.to_csv(outpath, index=False)
print(f"\nSaved results to {outpath}")

from google.colab import files
files.download(outpath)


Final results:
  Variable  MTI_member  MTI_nonmember  GETI_member  GETI_nonmember  \
0      rta         NaN       1.120702     1.569085        0.912470   
1     lang    1.197059       1.010101     1.335590        0.986173   
2   colony         NaN       1.002149     2.290359        0.997665   
3     home    0.006409            NaN     2.581649             NaN   

   Welf_member  Welf_nonmember  
0     0.914365        1.017808  
1     0.944232        1.002816  
2     0.848025        1.000453  
3     0.828156             NaN  

Saved results to /content/PTI_MTI_GETI_Welfare_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>